In [ ]:
# Install the Python libraries
!pip install cartopy imageio imageio-ffmpeg

In [ ]:
import json
from datetime import datetime, timezone

def create_xyzt_from_geojson(json_file, output_file):
    """
    Reads a GeoJSON file from the USGS and converts it to a simple
    x, y, z, t text file for GMT or other tools.
    """
    print(f"Reading data from '{json_file}'...")

    try:
        # Open and load the GeoJSON data
        with open(json_file, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The input file '{json_file}' was not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{json_file}'. Please check the file format.")
        return

    # Open the output file for writing
    with open(output_file, 'w') as f_out:
        # Iterate through each earthquake 'feature' in the JSON
        for feature in data['features']:
            try:
                # 1. Extract coordinates [longitude, latitude, depth]
                coords = feature['geometry']['coordinates']
                lon = coords[0]
                lat = coords[1]
                depth = coords[2]

                # 2. Extract time (Unix timestamp in milliseconds)
                time_ms = feature['properties']['time']

                # 3. Convert timestamp to a formatted string (YYYY-MM-DDTHH:MM:SS)
                # We divide by 1000 to convert milliseconds to seconds
                dt_object = datetime.fromtimestamp(time_ms / 1000, tz=timezone.utc)
                time_str = dt_object.strftime('%Y-%m-%dT%H:%M:%S')

                # 4. Write the formatted line to the output file
                f_out.write(f"{lon} {lat} {depth} {time_str}\n")

            except (KeyError, IndexError):
                print(f"Warning: Skipping a feature with unexpected data format.")
                continue

    print(f"Success! Data has been written to '{output_file}'.")


# --- Main execution ---
if __name__ == "__main__":
    input_json = 'query.json'
    output_xyzt = 'Quakes_filtered.xyzt'
    create_xyzt_from_geojson(input_json, output_xyzt)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from cartopy.feature import ShapelyFeature
import imageio
import os
from tqdm import tqdm

# --- 1. Configuration ---
DATA_FILE = 'Quakes_filtered.xyzt'
TECTONIC_MAP_URL = 'https://raw.githubusercontent.com/drtinkooo/myanmar-earthquake/main/Myanmar_Tectonic_Map_2011.geojson'
OUTPUT_DIR = 'frames'
VIDEO_FILE = 'Myanmar_Animation_2025_Final.mp4'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. Load and Prepare Data ---
print("Loading earthquake data...")
try:
    df = pd.read_csv(
        DATA_FILE,
        sep=r'\s+',
        header=None,
        names=['lon', 'lat', 'depth', 'time'],
        on_bad_lines='skip'
    )
    df.ffill(inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    print(f"Loaded {len(df)} events.")
except Exception as e:
    print(f"Error loading earthquake data file: {e}")
    exit()

# --- 3. Setup the Map and Color Scheme ---
print("Setting up the map with OSM and Tectonic overlays...")

# Try to load the tectonic lineament data from the URL
try:
    print(f"Fetching tectonic map from {TECTONIC_MAP_URL}...")
    tectonic_lines = gpd.read_file(TECTONIC_MAP_URL)
    print("Tectonic map loaded successfully.")
except Exception as e:
    print(f"Could not fetch or read the tectonic map GeoJSON. Error: {e}")
    print("Continuing without the tectonic overlay.")
    tectonic_lines = None

colors = ['red', 'orange', 'yellow', 'green', 'blue', 'purple']
depth_bins = [0, 25, 50, 100, 200, 400, 800]
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(depth_bins, cmap.N)

fig = plt.figure(figsize=(12, 8))
# Use the OSM tile service's projection
osm_tiles = cimgt.OSM()
ax = fig.add_subplot(1, 1, 1, projection=osm_tiles.crs)

min_lon, max_lon = df['lon'].min() - 1, df['lon'].max() + 1
min_lat, max_lat = df['lat'].min() - 1, df['lat'].max() + 1
ax.set_extent([min_lon, max_lon, min_lat, max_lat], crs=ccrs.PlateCarree())

# --- EDITED SECTION: Add Online Map Layers ---

# Layer 1 (Bottom): OpenStreetMap as the primary background
ax.add_image(osm_tiles, 8, zorder=0)

# Layer 2 (Middle): Tectonic lineaments from the GeoJSON file
if tectonic_lines is not None:
    tectonic_feature = ShapelyFeature(
        tectonic_lines['geometry'],
        crs=ccrs.PlateCarree(),
        edgecolor='#FF4500',
        facecolor='none',
        linewidth=1.5,
    )
    ax.add_feature(tectonic_feature, zorder=2)

# Layer 3: Add international borders for context
ax.add_feature(cfeature.BORDERS.with_scale('10m'), edgecolor='black', linestyle=':', linewidth=1.5, zorder=3)

gl = ax.gridlines(draw_labels=True, linewidth=1, color='black', alpha=0.3, linestyle='--')
gl.top_labels = False
gl.right_labels = False

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', pad=0.05, shrink=0.7)
cbar.set_label('Depth (km)', fontsize=12)

# --- 4. Generate Frames (The Animation Loop) ---
start_date = df['time'].min()
end_date = df['time'].max()
date_range = pd.date_range(start=start_date.date(), end=end_date.date(), freq='D')

print(f"Generating {len(date_range)} frames from {start_date.date()} to {end_date.date()}...")
frame_files = []
for i, current_date in enumerate(tqdm(date_range)):
    start_fade_date = current_date - pd.Timedelta(days=6)
    mask = (df['time'] >= start_fade_date) & (df['time'] <= current_date + pd.Timedelta(days=1))
    daily_df = df[mask]

    scatter_plots = []
    if not daily_df.empty:
        age_in_days = (current_date + pd.Timedelta(days=1) - daily_df['time']).dt.days
        alphas = 1.0 - (age_in_days / 7.0)
        alphas = alphas.clip(lower=0).values

        scatter = ax.scatter(
            daily_df['lon'], daily_df['lat'],
            s=50,
            c=daily_df['depth'],
            cmap=cmap,
            norm=norm,
            alpha=alphas,
            edgecolor='k',
            linewidth=0.5,
            transform=ccrs.PlateCarree(),
            zorder=10
        )
        scatter_plots.append(scatter)

    timestamp = ax.text(0.02, 0.95, current_date.strftime('%Y-%m-%d'),
                        transform=ax.transAxes, fontsize=14,
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    frame_filename = os.path.join(OUTPUT_DIR, f'frame_{i:03d}.png')
    plt.savefig(frame_filename, dpi=150, bbox_inches='tight')
    frame_files.append(frame_filename)

    timestamp.remove()
    for s in scatter_plots:
        s.remove()

# --- 5. Assemble the Video ---
print(f"Assembling video... saving to {VIDEO_FILE}")
with imageio.get_writer(VIDEO_FILE, fps=10) as writer:
    for filename in tqdm(frame_files):
        image = imageio.imread(filename)
        writer.append_data(image)

print(f"\nDone! Video saved as {VIDEO_FILE}")